# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import entropy
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import sys
from pathlib import Path

# Add src/ to path (once, so imports work)
sys.path.append(str(Path().resolve().parent / "src"))
# 
# Enable autoreload for Jupyter notebooks
%load_ext autoreload
%autoreload 2

from paths import DATA_DATASETS, DATA_EMBEDDINGS

Failed to read module file 'C:\Users\JuliusAdmin\AppData\Local\Programs\Python\Python312\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\JuliusAdmin\Documents\GitHub\Marketing-Analytics\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\JuliusAdmin\Documents\GitHub\Marketing-Analytics\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\JuliusAdmin\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bo

In [2]:
# Load data
transactions = pd.read_csv(DATA_DATASETS / "transactions.csv", sep=";")
outfits = pd.read_csv(DATA_DATASETS / "outfits.csv", sep=";")
outfit_clusters = pd.read_csv(DATA_DATASETS / "outfits_clusters_labels.csv", sep=";")
embeddings = pd.read_csv(DATA_EMBEDDINGS / "embeddings_voyage_multimodal.csv", sep=";")
test_customers = pd.read_csv(DATA_DATASETS / "test_customers.csv", sep=";")
baseline_pred = pd.read_csv(DATA_DATASETS / "pred.csv")


outfits = outfits.rename(columns={"id": "outfit.id"}) # Rename for consistency with transactions

transactions["rentalPeriod.start"] = pd.to_datetime(transactions["rentalPeriod.start"])
transactions["rentalPeriod.end"] = pd.to_datetime(transactions["rentalPeriod.end"])

# Join outfit meta data and cluster information
outfits_enriched = outfits.merge(
    outfit_clusters, left_on="outfit.id", right_on="outfit.id", how="left"
)

# Join transactions with outfit information
master = transactions.merge(
    outfits_enriched[["outfit.id", "cluster", "cluster_name", 
                       "pricePerWeek", "pricePerMonth", "retailPrice"]],
    on="outfit.id", how="left"
)

# Calculate rental duration in days
master["rentalPeriod.start"] = pd.to_datetime(master["rentalPeriod.start"])
master["rentalPeriod.end"]   = pd.to_datetime(master["rentalPeriod.end"])
master["duration_days"] = (master["rentalPeriod.end"] - master["rentalPeriod.start"]).dt.days

# Cleaning: Remove rentals with non-positive duration and missing price information
master = master[
    (master["duration_days"] > 0) &
    (
        (master["duration_days"] <= 7) & master["pricePerWeek"].notna() |
        (master["duration_days"] > 7)  & master["pricePerMonth"].notna()
    )
].copy()

# Revenue calculation based on duration and pricing
master["revenue"] = np.where(
    master["duration_days"] <= 7,
    master["pricePerWeek"],
    master["pricePerMonth"] * master["duration_days"] / 30
)

print(f"Master shape: {master.shape}")
print(master.head(2))

Master shape: (60897, 11)
   customer.id                                outfit.id rentalPeriod.start  \
0         3448  outfit.5c081909537b42239e465d2d615c705f         2023-03-26   
1         2924  outfit.c34969dd8b334064aa90bfb60c8ec308         2023-03-27   

  rentalPeriod.end  cluster         cluster_name  pricePerWeek  pricePerMonth  \
0       2023-04-25      3.0  Sweaters & Knitwear         750.0         1500.0   
1       2023-04-26      7.0      Jackets & Coats         990.0         1980.0   

   retailPrice  duration_days  revenue  
0       2500.0             30   1500.0  
1       3900.0             30   1980.0  


# CLV Model

### Feature Engineering

#### RFM Features
| Feature | Description |
|---|---|
| `recency` | Days since last rental before cutoff — low recency = recently active customer |
| `frequency` | Total number of rentals in training period |
| `monetary` | Total revenue generated across all rentals |
| `avg_revenue` | Average revenue per rental transaction |
| `std_revenue` | Variability in spending — high std = inconsistent renter |
| `weighted_rev` | Revenue weighted by exponential time decay (λ=365 days) — recent transactions count more |

#### Temporal Features
| Feature | Description |
|---|---|
| `tenure_days` | Days between first and last rental — captures customer lifetime so far |
| `avg_days_between_rentals` | Mean inter-rental interval — proxy for rental cadence |
| `recency_ratio` | Recency divided by avg_days_between_rentals — values above 1 indicate the customer is overdue relative to their personal rhythm; more informative than raw recency alone |
| `revenue_trend` | Revenue difference between second and first half of rental history — positive = growing customer, negative = declining customer |
| `pct_weekly_rentals` | Share of rentals with duration ≤7 days — distinguishes event-driven one-time renters from habit-based monthly renters |
| `active_months` | Number of distinct calendar months with at least one rental |
| `n_spring/summer/autumn/winter` | Rental count per season — captures seasonal preferences |

#### Product Cluster Affinity Features
| Feature | Description |
|---|---|
| `cluster_affinity_0..7` | Share of total spending allocated to each of the 8 product clusters — captures style preference profile per customer |
| `n_clusters_rented` | Number of distinct product clusters ever rented — operationalizes Hypothesis 3: customers who engage across multiple style categories show significantly higher loyalty than those concentrating on a single cluster |
| `style_entropy` | Shannon entropy across cluster affinity shares — continuous measure of style diversity; higher entropy = broader engagement across categories |

#### Price Segment Features
| Feature | Description |
|---|---|
| `avg_retail_price` | Average retail price of rented outfits — proxy for price sensitivity |
| `max_retail_price` | Highest retail price item ever rented — captures willingness to pay |
| `avg_price_per_week` | Average weekly rental rate across all transactions |

#### Optional: Embedding Features
| Feature | Description |
|---|---|
| `emb_dim_0..N` | Mean-pooled Voyage multimodal embeddings across all rented outfits per customer (2048-dim reduced via PCA) — captures latent style preferences beyond discrete clusters |

In [3]:
# Revenue trend: difference between second and first half of rental history
# Positive = growing customer, Negative = declining customer
# transaction-based trend 
# def revenue_trend(group):
#     group = group.sort_values("rentalPeriod.start")
#     mid = len(group) // 2
#     if mid == 0:
#         return 0.0
#     first_half  = group.iloc[:mid]["revenue"].sum()
#     second_half = group.iloc[mid:]["revenue"].sum()
#     return second_half - first_half

# time-based trend
def revenue_trend_time(group, end):
    group = group.sort_values("rentalPeriod.start")
    midpoint = end - (end - group["rentalPeriod.start"].min()) / 2

    first_half = group[group["rentalPeriod.start"] < midpoint]["revenue"].sum()
    second_half = group[group["rentalPeriod.start"] >= midpoint]["revenue"].sum()

    return second_half - first_half


def get_labels(df, start, end):
    """
    Computes actual revenue per customer in the label period [start, end).
    Customers with no activity in this period get 0.
    Returns a Series indexed by customer.id.
    """
    
    # Filter label period
    label_df = df[
        (df["rentalPeriod.start"] >= start) &
        (df["rentalPeriod.start"] <  end)
    ]
    
    # Aggregate revenue in label period
    labels = label_df.groupby("customer.id")["revenue"].sum()
    
    return labels


def build_features(df, end, start = None):
    """Build all customer features using only data before cutoff."""
    
    if start is not None:
        df = df[(df["rentalPeriod.start"] >= start) & (df["rentalPeriod.start"] < end)].copy()
    else:
        df = df[df["rentalPeriod.start"] < end].copy()

    df["month"] = df["rentalPeriod.start"].dt.month


    df["time_weight"] = np.exp(
        -(end - df["rentalPeriod.start"]).dt.days / 365
    )

    df["weighted_freq"] = df["time_weight"]

    weighted_freq = df.groupby("customer.id")["weighted_freq"].sum().reset_index()
    weighted_freq.columns = ["customer.id", "weighted_frequency"]

    # RFM
    rfm = df.groupby("customer.id").agg(
        recency        = ("rentalPeriod.start", lambda x: (end - x.max()).days),
        frequency      = ("rentalPeriod.start", "count"),
        monetary       = ("revenue", "sum"),
        avg_revenue    = ("revenue", "mean"),
        std_revenue    = ("revenue", "std"),
        avg_duration   = ("duration_days", "mean"),
        weighted_rev   = ("revenue", lambda x: (x * df.loc[x.index, "time_weight"]).sum()),
        first_rental   = ("rentalPeriod.start", "min"),
        last_rental    = ("rentalPeriod.start", "max"),
    ).reset_index()

    rfm["recency_x_monetary"] = rfm["recency"] * rfm["monetary"]
    rfm["recency_x_frequency"] = rfm["recency"] * rfm["frequency"]
    rfm["monetary_x_frequency"] = rfm["monetary"] * rfm["frequency"]

    rfm["tenure_days"]              = (rfm["last_rental"] - rfm["first_rental"]).dt.days
    rfm["avg_days_between_rentals"] = rfm["tenure_days"] / rfm["frequency"].clip(lower=1)
    rfm["recency_ratio"]            = rfm["recency"] / rfm["avg_days_between_rentals"].clip(lower=1)
    rfm["std_revenue"]              = rfm["std_revenue"].fillna(0)
    rfm = rfm.drop(columns=["first_rental", "last_rental"])

    rfm["revenue_per_day"] = rfm["monetary"] / rfm["tenure_days"].clip(lower=1)
    rfm["rentals_per_day"] = rfm["frequency"] / rfm["tenure_days"].clip(lower=1)

    # Seasonality
    seasonality = df.groupby("customer.id").agg(
        n_spring      = ("month", lambda x: x.isin([3, 4, 5]).sum()),
        n_summer      = ("month", lambda x: x.isin([6, 7, 8]).sum()),
        n_autumn      = ("month", lambda x: x.isin([9, 10, 11]).sum()),
        n_winter      = ("month", lambda x: x.isin([12, 1, 2]).sum()),
        active_months = ("month", lambda x: df.loc[x.index, "rentalPeriod.start"].dt.to_period("M").nunique()),
    ).reset_index()

    # Trend
    trend = df.groupby("customer.id").apply(lambda x: revenue_trend_time(x, end)).reset_index()
    trend.columns = ["customer.id", "revenue_trend"]

    # Weekly rentals
    pct_weekly = df.groupby("customer.id").apply(
        lambda x: (x["duration_days"] <= 7).mean()
    ).reset_index()
    pct_weekly.columns = ["customer.id", "pct_weekly_rentals"]

    # Cluster affinity
    cluster_affinity = (
        df.groupby(["customer.id", "cluster"])["revenue"]
        .sum()
        .unstack(fill_value=0)
    )
    cluster_affinity = cluster_affinity.div(cluster_affinity.sum(axis=1), axis=0)
    cluster_affinity.columns = [f"cluster_affinity_{int(c)}" for c in cluster_affinity.columns]
    cluster_affinity = cluster_affinity.reset_index()

    affinity_cols = [c for c in cluster_affinity.columns if c.startswith("cluster_affinity_")]
    cluster_affinity["n_clusters_rented"] = (cluster_affinity[affinity_cols] > 0).sum(axis=1)
    cluster_affinity["style_entropy"] = cluster_affinity[affinity_cols].apply(
        lambda row: entropy(row + 1e-9), axis=1
    )

    cluster_freq = (
    df.groupby(["customer.id", "cluster"])
    .size()
    .unstack(fill_value=0)
    )

    cluster_freq = cluster_freq.div(cluster_freq.sum(axis=1), axis=0)

    # Price features
    price_features = df.groupby("customer.id").agg(
        avg_retail_price   = ("retailPrice", "mean"),
        max_retail_price   = ("retailPrice", "max"),
        avg_price_per_week = ("pricePerWeek", "mean"),
    ).reset_index()

    features = rfm \
        .merge(seasonality,      on="customer.id", how="left") \
        .merge(trend,            on="customer.id", how="left") \
        .merge(pct_weekly,       on="customer.id", how="left") \
        .merge(cluster_affinity, on="customer.id", how="left") \
        .merge(price_features,   on="customer.id", how="left") \
        .merge(weighted_freq,    on="customer.id", how="left") \
        .merge(cluster_freq.add_prefix("cluster_freq_"), on="customer.id", how="left")

    return features

#### Use just data after clear break

In [4]:
# Fold structure
TIMEFRAME_START  = pd.Timestamp("2020-04-01")
TRAIN_CUTOFF     = pd.Timestamp("2021-09-06")  # features end here for training
LABEL_END        = pd.Timestamp("2022-09-06")  # labels end here for training
FINAL_CUTOFF     = pd.Timestamp("2023-09-06")  # features end here for submission

# Training fold
# Features: April 2020 → Sept 2021
# Labels:   Sept 2021 → Sept 2022  (genuinely future, never seen during training)
X_train = build_features(master, start=TIMEFRAME_START, end=TRAIN_CUTOFF)

y_raw = get_labels(master, start=TRAIN_CUTOFF, end=LABEL_END)
# Merge labels with X_train to ensure there are no customers in y_train that are not in X_train (and thus avoid leakage)
y_train = (
    X_train[["customer.id"]]
    .merge(y_raw.reset_index(), on="customer.id", how="left")
    ["revenue"].fillna(0).values
)

X_train_f = X_train.drop(columns=["customer.id"])

# Fit and evaluate
lgbm_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1
)

lgbm_model.fit(X_train_f, y_train)

train_predictions = np.clip(lgbm_model.predict(X_train_f), 0, None)
train_mae = mean_absolute_error(y_train, train_predictions)
print(f"MAE on training fold: {train_mae:.2f} NOK")


# Test fold
# Features: Sept 2021 → Sept 2022 (former training labels, now features)
# Labels:   Sept 2022 → Sept 2023
X_test   = build_features(master, start=TIMEFRAME_START, end=LABEL_END)

y_raw = get_labels(master, start=LABEL_END, end=FINAL_CUTOFF)
# Merge labels with X_test to ensure there are no customers in y_test that are not in X_test (and thus avoid leakage)
y_test = (
    X_test[["customer.id"]]
    .merge(y_raw.reset_index(), on="customer.id", how="left")
    ["revenue"].fillna(0).values
)

X_test_f = X_test.drop(columns=["customer.id"])

test_predictions = np.clip(lgbm_model.predict(X_test_f), 0, None)

test_mae = mean_absolute_error(y_test, test_predictions)
print(f"MAE on test fold: {test_mae:.2f} NOK")

MAE on training fold: 3237.41 NOK
MAE on test fold: 12671.09 NOK


In [5]:
# ── 1. Naive baseline comparison ─────────────────────────────────────────────
# Before doing anything else, check if the model is actually useful
mean_pred = np.full_like(y_test, y_train.mean(), dtype=float)
zero_pred  = np.zeros_like(y_test)

print(f"MAE (predict zero for everyone): {mean_absolute_error(y_test, zero_pred):.2f} NOK")
print(f"MAE (predict mean for everyone): {mean_absolute_error(y_test, mean_pred):.2f} NOK")
print(f"MAE (LightGBM):                  {test_mae:.2f} NOK")

MAE (predict zero for everyone): 13389.62 NOK
MAE (predict mean for everyone): 21073.53 NOK
MAE (LightGBM):                  12671.09 NOK


In [6]:
# ── 2. Where is the error coming from? ───────────────────────────────────────
residuals = pd.DataFrame({
    "customer.id": X_test["customer.id"].values,
    "actual":      y_test,
    "predicted":   test_predictions,
    "error":       np.abs(y_test - test_predictions)
})

# Error by actual revenue bucket
residuals["bucket"] = pd.cut(
    residuals["actual"],
    bins   = [0, 1, 1000, 5000, 20000, 999999],
    labels = ["zero (churned)", "1–1k NOK", "1k–5k NOK", "5k-20k NOK", "20k+ NOK"],
    include_lowest=True)

print("=== MAE by actual revenue bucket ===")
print(residuals.groupby("bucket", observed=True).agg(
    n_customers = ("actual", "count"),
    mae         = ("error", "mean"),
    avg_actual  = ("actual", "mean"),
    avg_pred    = ("predicted", "mean")
).round(2))

=== MAE by actual revenue bucket ===
                n_customers       mae  avg_actual  avg_pred
bucket                                                     
zero (churned)         1088   6331.40        0.00   6331.40
1k–5k NOK                74  21300.06     2990.24  23515.17
5k-20k NOK              144  21478.80    12008.77  29151.66
20k+ NOK                356  26689.97    57030.92  44655.36


In [7]:
# ── 3. Is the model over- or underpredicting? ─────────────────────────────────
print(f"\nActual mean revenue:    {y_test.mean():.2f} NOK")
print(f"Predicted mean revenue: {test_predictions.mean():.2f} NOK")

# How many churned customers does the model correctly predict as zero?
churned        = residuals[residuals["actual"] == 0]
active         = residuals[residuals["actual"] >  0]

print(f"\nChurned customers (actual=0):  {len(churned)}")
print(f"  → avg prediction for them:   {churned['predicted'].mean():.2f} NOK")
print(f"\nActive customers (actual>0):   {len(active)}")
print(f"  → avg prediction for them:   {active['predicted'].mean():.2f} NOK")
print(f"  → avg actual for them:       {active['actual'].mean():.2f} NOK")


Actual mean revenue:    13389.62 NOK
Predicted mean revenue: 17282.69 NOK

Churned customers (actual=0):  1088
  → avg prediction for them:   6331.40 NOK

Active customers (actual>0):   574
  → avg prediction for them:   38040.54 NOK
  → avg actual for them:       38769.25 NOK


In [8]:
# ── 4. Feature importance ─────────────────────────────────────────────────────
importance = pd.DataFrame({
    "feature":    X_train_f.columns,
    "importance": lgbm_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== Top 15 features ===")
print(importance.head(15).to_string(index=False))

print("\n=== Bottom 10 features (candidates to drop) ===")
print(importance.tail(10).to_string(index=False))


=== Top 15 features ===
           feature  importance
     revenue_trend         330
           recency         326
weighted_frequency         277
      avg_duration         273
       avg_revenue         242
     style_entropy         242
       std_revenue         224
      weighted_rev         211
   revenue_per_day         209
     recency_ratio         205
  avg_retail_price         170
recency_x_monetary         157
avg_price_per_week         154
       tenure_days         139
cluster_affinity_2         138

=== Bottom 10 features (candidates to drop) ===
           feature  importance
cluster_affinity_0          41
cluster_affinity_4          39
  cluster_freq_5.0          36
  cluster_freq_7.0          36
pct_weekly_rentals          31
  cluster_freq_4.0          23
   rentals_per_day          21
  cluster_freq_0.0          14
 n_clusters_rented          12
     active_months           7


#### Use all of the data

In [9]:
# Fold structure
# TIMEFRAME_START  = pd.Timestamp("2017-04-01")
TRAIN_CUTOFF     = pd.Timestamp("2021-09-06")  # features end here for training
LABEL_END        = pd.Timestamp("2022-09-06")  # labels end here for training
FINAL_CUTOFF     = pd.Timestamp("2023-09-06")  # features end here for submission

# Training fold
# Features: Start → Sept 2021
# Labels:   Sept 2021 → Sept 2022  (genuinely future, never seen during training)
X_train = build_features(master, end=TRAIN_CUTOFF)

y_raw = get_labels(master, start=TRAIN_CUTOFF, end=LABEL_END)
# Merge labels with X_train to ensure there are no customers in y_train that are not in X_train (and thus avoid leakage)
y_train = (
    X_train[["customer.id"]]
    .merge(y_raw.reset_index(), on="customer.id", how="left")
    ["revenue"].fillna(0).values
)

X_train_f = X_train.drop(columns=["customer.id"])


# Test fold
# Features: Start → Sept 2022 (former training labels, now features)
# Labels:   Sept 2022 → Sept 2023
X_test   = build_features(master, end=LABEL_END)

y_raw = get_labels(master, start=LABEL_END, end=FINAL_CUTOFF)
# Merge labels with X_test to ensure there are no customers in y_test that are not in X_test (and thus avoid leakage)
y_test = (
    X_test[["customer.id"]]
    .merge(y_raw.reset_index(), on="customer.id", how="left")
    ["revenue"].fillna(0).values
)

X_test_f = X_test.drop(columns=["customer.id"])

In [10]:
# Fit and evaluate
lgbm_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1
)

# Train fold
lgbm_model.fit(X_train_f, y_train)

train_predictions = np.clip(lgbm_model.predict(X_train_f), 0, None)
train_mae = mean_absolute_error(y_train, train_predictions)
print(f"MAE on training fold: {train_mae:.2f} NOK")


# Test fold
test_predictions = np.clip(lgbm_model.predict(X_test_f), 0, None)

test_mae = mean_absolute_error(y_test, test_predictions)
print(f"MAE on test fold: {test_mae:.2f} NOK")

MAE on training fold: 850.03 NOK
MAE on test fold: 3309.84 NOK


With this model we can already beat the baseline.

In [11]:
# ── 2. Where is the error coming from? ───────────────────────────────────────
residuals = pd.DataFrame({
    "customer.id": X_test["customer.id"].values,
    "actual":      y_test,
    "predicted":   test_predictions,
    "error":       np.abs(y_test - test_predictions)
})

# Error by actual revenue bucket
residuals["bucket"] = pd.cut(
    residuals["actual"],
    bins   = [0, 1, 1000, 5000, 20000, 999999],
    labels = ["zero (churned)", "1–1k NOK", "1k–5k NOK", "5k-20k NOK", "20k+ NOK"],
    include_lowest=True)

print("=== MAE by actual revenue bucket ===")
print(residuals.groupby("bucket", observed=True).agg(
    n_customers = ("actual", "count"),
    mae         = ("error", "mean"),
    avg_actual  = ("actual", "mean"),
    avg_pred    = ("predicted", "mean")
).round(2))

=== MAE by actual revenue bucket ===
                n_customers       mae  avg_actual  avg_pred
bucket                                                     
zero (churned)         6193   1176.68        0.00   1176.68
1k–5k NOK                84  19088.36     2951.35  20616.39
5k-20k NOK              164  20738.02    11888.91  25326.28
20k+ NOK                368  27839.92    56699.01  43076.80


In [12]:
# ── 4. Feature importance ─────────────────────────────────────────────────────
importance = pd.DataFrame({
    "feature":    X_train_f.columns,
    "importance": lgbm_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== Top 15 features ===")
print(importance.head(15).to_string(index=False))

print("\n=== Bottom 10 features (candidates to drop) ===")
print(importance.tail(10).to_string(index=False))


=== Top 15 features ===
            feature  importance
            recency         440
        std_revenue         354
 cluster_affinity_3         311
      recency_ratio         296
       weighted_rev         278
      style_entropy         254
      revenue_trend         238
       avg_duration         229
 recency_x_monetary         220
        avg_revenue         205
 cluster_affinity_1         197
recency_x_frequency         194
   cluster_freq_3.0         194
        tenure_days         193
           n_summer         192

=== Bottom 10 features (candidates to drop) ===
           feature  importance
     active_months          57
pct_weekly_rentals          55
  cluster_freq_7.0          51
cluster_affinity_0          39
  cluster_freq_5.0          34
   rentals_per_day          28
  cluster_freq_4.0          24
cluster_affinity_4          22
 n_clusters_rented          17
  cluster_freq_0.0          15


#### Use of all data & log transforming revenue

In [13]:
# Fit and evaluate
lgbm_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1
)

y_train_log = np.log1p(y_train)  # log(1 + y) so 0 stays 0

lgbm_model.fit(X_train_f, y_train_log)

# Back transformation for MAE in NOK
predictions_log = lgbm_model.predict(X_test_f)
predictions_nok = np.expm1(predictions_log)  # exp(x) - 1
predictions_nok = np.clip(predictions_nok, 0, None)

mae = mean_absolute_error(y_test, predictions_nok)
print(f"MAE on test fold: {mae:.2f} NOK")

MAE on test fold: 3291.73 NOK


#### Use of all data and Tweedie regression with log-link

In [14]:
# Fit and evaluate
lgbm_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1,

    objective="tweedie",
    tweedie_variance_power=1.3
)

# Train fold
lgbm_model.fit(X_train_f, y_train)

train_predictions = np.clip(lgbm_model.predict(X_train_f), 0, None)
train_mae = mean_absolute_error(y_train, train_predictions)
print(f"MAE on training fold: {train_mae:.2f} NOK")


# Test fold
test_predictions = np.clip(lgbm_model.predict(X_test_f), 0, None)

test_mae = mean_absolute_error(y_test, test_predictions)
print(f"MAE on test fold: {test_mae:.2f} NOK")

MAE on training fold: 327.71 NOK
MAE on test fold: 2627.55 NOK


In [15]:
importance = pd.DataFrame({
    "feature":    X_train_f.columns,
    "importance": lgbm_model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

                     feature  importance
0                    recency        1299
6               weighted_rev         881
32          avg_retail_price         871
7         recency_x_monetary         750
5               avg_duration         595
3                avg_revenue         488
35        weighted_frequency         482
34        avg_price_per_week         459
12             recency_ratio         446
8        recency_x_frequency         435
4                std_revenue         399
33          max_retail_price         374
13           revenue_per_day         363
2                   monetary         352
20             revenue_trend         311
10               tenure_days         231
24        cluster_affinity_2         222
31             style_entropy         196
16                  n_summer         191
11  avg_days_between_rentals         190
29        cluster_affinity_7         158
38          cluster_freq_2.0         132
21        pct_weekly_rentals         129
23        cluste

### Feature ablation test

In [16]:
full_features = X_train_f.columns.tolist()

lgbm_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1,

    objective="tweedie",
    tweedie_variance_power=1.3
)

lgbm_model.fit(X_train_f, y_train)

baseline_pred = lgbm_model.predict(X_test_f)
baseline_pred = np.clip(baseline_pred, 0, None)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
print("Baseline MAE:", baseline_mae)

Baseline MAE: 2627.5495314281766


In [17]:
# Define feature groups for ablation and interpretability
feature_groups = {
    "rfm_core": ["recency", "frequency", "monetary"],
    
    "revenue_stats": ["avg_revenue", "std_revenue", "weighted_rev"],
    
    "time_features": ["recency_ratio", "avg_days_between_rentals", "tenure_days"],
    
    "velocity": ["revenue_per_day", "rentals_per_day"],
    
    "interactions": ["recency_x_monetary", "recency_x_frequency", "monetary_x_frequency"],
    
    "price": ["avg_retail_price", "max_retail_price", "avg_price_per_week"],
    
    "seasonality": ["n_spring", "n_summer", "n_autumn", "n_winter", "active_months"],
    
    "clusters_revenue": [c for c in full_features if "cluster_affinity_" in c],
    
    "clusters_freq": [c for c in full_features if "cluster_freq_" in c],
    
    "behavior": ["pct_weekly_rentals", "revenue_trend", "avg_duration"]
}

In [18]:
results = []

for group_name, cols in feature_groups.items():

    remaining_features = [f for f in full_features if f not in cols]

    
    X_train_sub = X_train_f[remaining_features]
    X_test_sub  = X_test_f[remaining_features]
    
    lgbm_model = lgb.LGBMRegressor(
        n_estimators      = 500,
        learning_rate     = 0.05,
        max_depth         = 6,
        min_child_samples = 20,
        subsample         = 0.8,
        random_state      = 42,
        verbose           = -1,

        objective="tweedie",
        tweedie_variance_power=1.3
    )
    lgbm_model.fit(X_train_sub, y_train)
    
    pred = lgbm_model.predict(X_test_sub)
    pred = np.clip(pred, 0, None)
    
    mae = mean_absolute_error(y_test, pred)
    
    results.append({
        "group_removed": group_name,
        "mae": mae,
        "delta_vs_baseline": mae - baseline_mae
    })


In [19]:
results_df = pd.DataFrame(results).sort_values("delta_vs_baseline")
results_df

,group_removed,mae,delta_vs_baseline
4,interactions,2565.494994,-62.054537
5,price,2590.257902,-37.291630
8,clusters_freq,2595.248429,-32.301102
9,behavior,2604.133807,-23.415725
7,clusters_revenue,2628.693031,1.143500
1,revenue_stats,2630.885400,3.335868
3,velocity,2631.757264,4.207732
2,time_features,2656.561381,29.011849
0,rfm_core,2658.178265,30.628734
6,seasonality,2660.706348,33.156817


In [20]:
fine_groups = {
    # interactions
    "recency_x_monetary": ["recency_x_monetary"],
    "recency_x_frequency": ["recency_x_frequency"],
    "monetary_x_frequency": ["monetary_x_frequency"],

    # price
    "avg_retail_price": ["avg_retail_price"],
    "max_retail_price": ["max_retail_price"],
    "avg_price_per_week": ["avg_price_per_week"],

    # behavior
    "pct_weekly_rentals": ["pct_weekly_rentals"],
    "revenue_trend": ["revenue_trend"],
    "avg_duration": ["avg_duration"],
}

In [21]:
results = []

for col in [c for c in full_features if "cluster_freq_" in c]:
    fine_groups[col] = [col]

for group_name, cols in fine_groups.items():

    remaining_features = [f for f in full_features if f not in cols]

    
    X_train_sub = X_train_f[remaining_features]
    X_test_sub  = X_test_f[remaining_features]
    
    lgbm_model = lgb.LGBMRegressor(
        n_estimators      = 500,
        learning_rate     = 0.05,
        max_depth         = 6,
        min_child_samples = 20,
        subsample         = 0.8,
        random_state      = 42,
        verbose           = -1,

        objective="tweedie",
        tweedie_variance_power=1.3
    )
    lgbm_model.fit(X_train_sub, y_train)
    
    pred = lgbm_model.predict(X_test_sub)
    pred = np.clip(pred, 0, None)
    
    mae = mean_absolute_error(y_test, pred)
    
    results.append({
        "group_removed": group_name,
        "mae": mae,
        "delta_vs_baseline": mae - baseline_mae
    })

In [22]:
results_df = pd.DataFrame(results).sort_values("delta_vs_baseline")
results_df

,group_removed,mae,delta_vs_baseline
8,avg_duration,2599.630713,-27.918819
1,recency_x_frequency,2600.816824,-26.732708
3,avg_retail_price,2608.793656,-18.755876
11,cluster_freq_2.0,2609.024204,-18.525327
14,cluster_freq_5.0,2609.069699,-18.479833
15,cluster_freq_6.0,2615.131784,-12.417748
6,pct_weekly_rentals,2617.202473,-10.347058
5,avg_price_per_week,2629.461733,1.912201
13,cluster_freq_4.0,2633.886268,6.336737
7,revenue_trend,2637.459946,9.910414
